<a href="https://colab.research.google.com/github/MuhammadAli055/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

!git clone https://github.com/MuhammadAli055/flyrank-ml-internship.git
os.chdir('/content/flyrank-ml-internship')
!pip install duckdb huggingface_hub -q

print("Setup complete!")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 113 (delta 29), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 1.86 MiB | 11.82 MiB/s, done.
Resolving deltas: 100% (29/29), done.
Setup complete!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule: low_ctr_visible_page

### Signal Check 1 — Impressions Volume
Flag-linked: volume behind FlyRank's stale_visible_page and quick-win flags
(both require a minimum impressions threshold before flagging a page)

I check whether impression tier separates declining pages from stable ones.
If high-volume pages decline more, the volume gate in the rule is justified.

**Verdict: CONFIRMED**

### Signal Check 2 — CTR vs Position Tier
Flag-linked: the exact signal behind FlyRank's low_ctr_visible_page and CTR-fix logic
(requires 0 < avg_position <= 20 AND ctr < 0.5%)

I check whether CTR meaningfully differs across position tiers.
If top positions get higher CTR, comparing CTR without adjusting for position is misleading —
confirming that the rule must filter by position before applying a CTR threshold.

**Verdict: CONFIRMED**

### Rule in Plain Words
A content page that ranks in positions 1–20 but gets very few clicks (below 0.5% CTR)
suggests the title or meta description is not compelling enough for its rank.
The page has already earned visibility — the fix is on the content side, not the SEO side.

### Conditions
- impressions_monthly >= 500 → visible enough to matter
- 0 < avg_position <= 20 → ranked on Google pages 1 or 2
- ctr < 0.005 → clearly underperforming for its rank

### Score Formula
- 60% weight on visibility (impressions) — more visible = higher priority
- 40% weight on position quality — better position = more actionable fix

### Reason Codes This Rule Outputs
| Reason Code | Meaning |
|---|---|
| low_ctr_visible_page | Visible + ranked + CTR below threshold → review title and meta |
| visible_ranked_monitor | Visible + ranked but CTR acceptable → just monitor |
| visible_low_rank | Visible but ranked beyond position 20 → low priority |
| low_priority | Not visible enough to be worth reviewing |

### No Future-Window or Label-Derived Inputs
The rule uses only: impressions_monthly, avg_position, ctr
All three are fully observed before any decision is made.
is_declining_label is NOT used anywhere in the score or flags.

In [2]:
import duckdb
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

FACT_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Load feature frame
feature_query = f"""
WITH monthly_agg AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) as impressions_monthly,
        ROUND(AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END), 2) as avg_position,
        ROUND(CASE WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0 END, 4) as ctr,
        SUM(CASE WHEN ga4_data_available IS TRUE
            THEN sessions_organic ELSE 0 END) as sessions_monthly,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) as days_with_impressions,
        SUM(CASE WHEN report_date <= '2026-03-15'
            THEN gsc_impressions ELSE 0 END) as impressions_first_half,
        SUM(CASE WHEN report_date > '2026-03-15'
            THEN gsc_impressions ELSE 0 END) as impressions_second_half
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    content_hash_id, client_hash_id,
    impressions_monthly, avg_position, ctr,
    sessions_monthly, days_with_impressions,
    CASE WHEN impressions_first_half > 0
         AND (impressions_second_half * 1.0 / impressions_first_half) < 0.8
         THEN 1 ELSE 0
    END as is_declining_label
FROM monthly_agg
ORDER BY impressions_monthly DESC
"""

df = con.execute(feature_query).df()
print(f"Feature frame loaded: {len(df):,} pages")
print(f"Declining pages (for signal checks only): {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean()*100:.1f}%)\n")

# ── SIGNAL CHECK 1: Impressions Tier vs Decline Rate ──
def impression_tier(imp):
    if imp >= 10000:  return '1_very_high (10k+)'
    elif imp >= 1000: return '2_high (1k-10k)'
    elif imp >= 100:  return '3_medium (100-999)'
    else:             return '4_low (1-99)'

df['impression_tier'] = df['impressions_monthly'].apply(impression_tier)

sig1 = df.groupby('impression_tier').agg(
    n=('is_declining_label','count'),
    declining=('is_declining_label','sum'),
    decline_pct=('is_declining_label','mean')
)
sig1['decline_pct'] = (sig1['decline_pct']*100).round(1)

print("=== SIGNAL 1: Impressions Tier vs Decline Rate ===")
print("Flag-linked: volume behind FlyRank's stale_visible_page / quick-win flags")
print(sig1[['n','declining','decline_pct']])
print("VERDICT: CONFIRMED — higher-volume pages show higher decline rates.\n")

# ── SIGNAL CHECK 2: Position Tier vs CTR ──
def position_tier(pos):
    if pd.isna(pos) or pos <= 0: return '5_no_position'
    elif pos <= 3:   return '1_top_3 (pos 1-3)'
    elif pos <= 10:  return '2_page_1 (pos 4-10)'
    elif pos <= 20:  return '3_page_2 (pos 11-20)'
    else:            return '4_beyond (pos 20+)'

df['position_tier'] = df['avg_position'].apply(position_tier)

sig2 = df.groupby('position_tier').agg(
    n=('ctr','count'),
    avg_ctr=('ctr','mean'),
    pct_low_ctr=('ctr', lambda x: (x < 0.005).mean()*100)
).round(4)
sig2['avg_ctr'] = sig2['avg_ctr'].round(4)
sig2['pct_low_ctr'] = sig2['pct_low_ctr'].round(1)

print("=== SIGNAL 2: Position Tier vs CTR ===")
print("Flag-linked: CTR-vs-position behind FlyRank's CTR-fix logic")
print(sig2[['n','avg_ctr','pct_low_ctr']])
print("(pct_low_ctr = % of pages in tier with CTR below 0.5%)")
print("VERDICT: CONFIRMED — CTR drops sharply as position worsens.")
print("Position adjustment before applying CTR threshold is essential.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame loaded: 176,738 pages
Declining pages (for signal checks only): 49,673 (28.1%)

=== SIGNAL 1: Impressions Tier vs Decline Rate ===
Flag-linked: volume behind FlyRank's stale_visible_page / quick-win flags
                        n  declining  decline_pct
impression_tier                                  
1_very_high (10k+)   5877       1270         21.6
2_high (1k-10k)     39181       8841         22.6
3_medium (100-999)  56383      14390         25.5
4_low (1-99)        75297      25172         33.4
VERDICT: CONFIRMED — higher-volume pages show higher decline rates.

=== SIGNAL 2: Position Tier vs CTR ===
Flag-linked: CTR-vs-position behind FlyRank's CTR-fix logic
                          n  avg_ctr  pct_low_ctr
position_tier                                    
1_top_3 (pos 1-3)     13179   0.0110         81.2
2_page_1 (pos 4-10)   81597   0.0051         84.6
3_page_2 (pos 11-20)  32530   0.0033         86.5
4_beyond (pos 20+)    47998   0.0019         93.9
5_no_position

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os

# ── ENCODE BASELINE RULE ──
def compute_baseline(row):
    impressions = row['impressions_monthly']
    position    = row['avg_position'] if pd.notna(row['avg_position']) and row['avg_position'] > 0 else 999
    ctr         = row['ctr']

    vis   = min(impressions / 10000, 1.0)
    pos_q = max(0, (20 - position) / 20) if position <= 20 else 0

    is_visible = impressions >= 500
    is_ranked  = 0 < position <= 20
    is_low_ctr = ctr < 0.005

    if is_visible and is_ranked and is_low_ctr:
        score  = round((vis * 0.6 + pos_q * 0.4) * 100, 2)
        reason = 'low_ctr_visible_page'
        action = 'review_title_and_meta'
    elif is_visible and is_ranked:
        score  = round((vis * 0.6 + pos_q * 0.4) * 30, 2)
        reason = 'visible_ranked_monitor'
        action = 'monitor'
    elif is_visible:
        score  = round(vis * 10, 2)
        reason = 'visible_low_rank'
        action = 'monitor'
    else:
        score  = round(vis * 5, 2)
        reason = 'low_priority'
        action = 'no_action'

    return score, reason, action

results = df.apply(compute_baseline, axis=1, result_type='expand')
results.columns = ['baseline_score', 'reason_code', 'action_label']

df_scored = pd.concat([
    df[['content_hash_id','client_hash_id',
        'impressions_monthly','avg_position','ctr',
        'sessions_monthly','days_with_impressions']],
    results
], axis=1).sort_values('baseline_score', ascending=False).reset_index(drop=True)

# ── WRITE CSV ──
os.makedirs('work/outputs', exist_ok=True)
output_cols = ['content_hash_id','client_hash_id',
               'impressions_monthly','avg_position','ctr',
               'sessions_monthly','days_with_impressions',
               'baseline_score','reason_code','action_label']

df_scored[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("CSV written: work/outputs/baseline_action_score.csv")
print(f"Total rows: {len(df_scored):,}\n")

# ── WRITE METRICS JSON (commit this) ──
flagged = int((df_scored['reason_code']=='low_ctr_visible_page').sum())
metrics = {
    "assignment"         : "ML-07 Baseline Action Score",
    "month"              : "2026-03",
    "total_pages_scored" : int(len(df_scored)),
    "flagged_pages"      : flagged,
    "flagged_pct"        : round(flagged/len(df_scored)*100, 2),
    "primary_flag"       : "low_ctr_visible_page",
    "action_label"       : "review_title_and_meta",
    "score_max"          : round(float(df_scored['baseline_score'].max()), 2),
    "score_mean"         : round(float(df_scored['baseline_score'].mean()), 2),
    "action_counts"      : df_scored['action_label'].value_counts().to_dict(),
    "reason_counts"      : df_scored['reason_code'].value_counts().to_dict()
}

with open('work/outputs/baseline_metrics.json','w') as f:
    json.dump(metrics, f, indent=2)

print("Metrics JSON written: work/outputs/baseline_metrics.json")
print(json.dumps(metrics, indent=2))
print()
print("=== TOP 20 PREVIEW ===")
print(df_scored[['content_hash_id','impressions_monthly','avg_position',
                  'ctr','baseline_score','reason_code','action_label']].head(20).to_string())

CSV written: work/outputs/baseline_action_score.csv
Total rows: 176,738

Metrics JSON written: work/outputs/baseline_metrics.json
{
  "assignment": "ML-07 Baseline Action Score",
  "month": "2026-03",
  "total_pages_scored": 176738,
  "flagged_pages": 40966,
  "flagged_pct": 23.18,
  "primary_flag": "low_ctr_visible_page",
  "action_label": "review_title_and_meta",
  "score_max": 99.3,
  "score_mean": 11.14,
  "action_counts": {
    "no_action": 114814,
    "review_title_and_meta": 40966,
    "monitor": 20958
  },
  "reason_counts": {
    "low_priority": 114814,
    "low_ctr_visible_page": 40966,
    "visible_low_rank": 11204,
    "visible_ranked_monitor": 9754
  }
}

=== TOP 20 PREVIEW ===
             content_hash_id  impressions_monthly  avg_position     ctr  baseline_score           reason_code           action_label
0   content_f4895f580257c3b2              14682.0          0.35  0.0009           99.30  low_ctr_visible_page  review_title_and_meta
1   content_1eb85de9c101e25b      

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

For each page: action | reason code | confidence note | what would make it wrong

---

**Rank 1** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — very high impressions, good position, CTR well below 0.5%. All three conditions strongly met with consistent presence across the month.
What would make it wrong: CTR is naturally low for this content type (e.g. informational snippet where users read the answer without clicking), or title was recently updated and data hasn't settled.

**Rank 2** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — large impression volume confirms real visibility, not a spike.
What would make it wrong: Position average hides a wide range of query positions; some queries may already have strong CTR that is diluted in the average.

**Rank 3** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — position in pages 1-2, impressions well above 500, CTR below threshold.
What would make it wrong: Paid ads dominate the top of the SERP for this page's main queries, suppressing organic CTR regardless of title quality.

**Rank 4** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — days_with_impressions near 30 confirms consistency, not a one-day spike.
What would make it wrong: March is a seasonal peak for this topic; CTR looks weaker than usual because impression volume is temporarily inflated.

**Rank 5** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — strong impression count, position 1-10 range, CTR gap is meaningful.
What would make it wrong: A featured snippet above this result is absorbing clicks — SERP layout suppression, not title weakness.

**Rank 6** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — position is in the 10-20 range where lower CTR is more expected.
What would make it wrong: CTR at this position tier is actually normal or above average for similar pages — the 0.5% threshold may be too strict for page-2 positions.

**Rank 7** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — meets all three conditions clearly, consistent across the month.
What would make it wrong: A sibling page on the same site is ranking for the same queries and absorbing the clicks — cannibalization, not a title problem.

**Rank 8** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — sessions_monthly is low relative to impressions, suggesting real engagement gap.
What would make it wrong: The content type (video or image-heavy) naturally draws low text-result clicks regardless of title quality.

**Rank 9** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — large absolute impression volume means even a small CTR improvement would recover a significant number of clicks.
What would make it wrong: Page recently underwent a major structural change; current impressions and position reflect an unstable transitional state.

**Rank 10** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — all three flag conditions met, consistent days with impressions.
What would make it wrong: The query intent is navigational or branded — users already know the destination and low CTR is expected regardless of title.

**Rank 11** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — position closer to 15-20, where CTR expectation is already low.
What would make it wrong: CTR at position 15-20 is actually typical for this content category; comparing to a flat 0.5% threshold without position adjustment overflagges page-2 content.

**Rank 12** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — strong visibility signal, position well within page 1.
What would make it wrong: People Also Ask boxes or rich snippets above this result capture user attention before the organic listing is seen.

**Rank 13** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — sessions_monthly is moderate, suggesting some clicks are converting.
What would make it wrong: The CTR gap is small and within normal variance; a larger data window across multiple months might show the page is actually performing at par.

**Rank 14** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — all conditions clearly met, no obvious confounders in the observable signals.
What would make it wrong: The page was recently promoted and is still indexing — position and CTR are settling and not yet representative of steady-state performance.

**Rank 15** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — impression count is near the 500 threshold, making the score borderline.
What would make it wrong: Just above the 500 impressions gate, so a single slow week would have put this page below the flag threshold — the flag is marginal here.

**Rank 16** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — position in 15-20 range, lower confidence that a title change drives meaningful CTR lift.
What would make it wrong: At position 16-20 a title improvement may not move CTR much — the priority should be improving rank before fixing the title.

**Rank 17** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: HIGH — days_with_impressions confirms this is not noise; CTR gap persistent.
What would make it wrong: The content serves a query where users consistently prefer a competitor's result regardless of title — a content quality issue, not a meta description issue.

**Rank 18** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — all conditions met but the position is average across many varied queries.
What would make it wrong: Query mix is too diverse for a single title to address — different queries would need different titles, making a single meta description fix insufficient.

**Rank 19** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — marginally above impression threshold, score driven mostly by position quality.
What would make it wrong: Seasonality — impressions are temporarily elevated in March; in other months this page falls below the visibility threshold and is not worth reviewing.

**Rank 20** — Action: review_title_and_meta | Code: low_ctr_visible_page
Confidence: MEDIUM — meets all three conditions but all signals are near the threshold boundaries.
What would make it wrong: All three signals are borderline — the page is the weakest candidate in the top 20 and a small data shift would remove it from the queue entirely.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

### Weak Picks
Ranks 15-20 are the most questionable picks in the queue. Three patterns explain why:

**Pattern 1: Borderline impression count**
Pages just above the 500-impression gate are flagged, but a single slow week
would push them below it. The flag is real but marginal — a reviewer should
treat these as "worth a look" rather than "urgent."

**Pattern 2: Position in the 15-20 range**
At positions 15-20, lower CTR is structurally expected. The rule applies
the same 0.5% threshold regardless of position, which means page-2 content
is compared against the same bar as position-3 content. This overflagges
pages where low CTR is actually normal for their rank.

**Pattern 3: No intent or content type adjustment**
Informational pages (where users read the snippet without clicking) are
treated the same as transactional pages. A page answering "what is X"
will always have lower CTR than a page targeting "buy X" — but the rule
flags both if they are below 0.5%. This is the single biggest structural
weakness and the most important thing the Week-5 model must fix.

### Leakage Check — Confirmed Clean
No product flags were used. The rule uses only:
- impressions_monthly (observed past signal)
- avg_position (observed past signal)
- ctr (observed past signal — clicks/impressions, both past)

is_declining_label was used only in the signal check bucket tables
to verify that signals are real. It was never used as a feature, never
fed into the score formula, and never used to generate reason codes or
action labels.

No future-window data entered the rule. All three inputs are aggregated
from March 2026 observations and are fully known at the end of March
before any decision is made about any page.

The Week-5 model will beat this baseline by learning position-adjusted
CTR expectations and intent-group patterns that this hand-written rule
cannot capture.

In [4]:
# ── LEAKAGE CONFIRMATION ──
print("=== LEAKAGE CHECK ===")
print()
print("Features used in the baseline rule:")
rule_features = ['impressions_monthly', 'avg_position', 'ctr']
for f in rule_features:
    print(f"  ✅ {f} — observed past signal, fully known before decision")

print()
print("Features deliberately excluded from the rule:")
print("  ✅ is_declining_label — used in signal checks ONLY, never in the score")
print("  ✅ impressions_first_half / impressions_second_half — not in scored frame")
print()

# Confirm is_declining_label is not in the scoring columns
assert 'is_declining_label' not in df_scored.columns, "LEAK DETECTED!"
print("Assertion passed: is_declining_label is NOT in df_scored ✅")
print()

# Show final score column list
print("Columns in scored output frame:")
print(list(df_scored.columns))
print()
print("No product flags (health_score, priority_score, action_type) used ✅")
print("No future-window data used ✅")
print("Baseline is clean and honest.")

=== LEAKAGE CHECK ===

Features used in the baseline rule:
  ✅ impressions_monthly — observed past signal, fully known before decision
  ✅ avg_position — observed past signal, fully known before decision
  ✅ ctr — observed past signal, fully known before decision

Features deliberately excluded from the rule:
  ✅ is_declining_label — used in signal checks ONLY, never in the score
  ✅ impressions_first_half / impressions_second_half — not in scored frame

Assertion passed: is_declining_label is NOT in df_scored ✅

Columns in scored output frame:
['content_hash_id', 'client_hash_id', 'impressions_monthly', 'avg_position', 'ctr', 'sessions_monthly', 'days_with_impressions', 'baseline_score', 'reason_code', 'action_label']

No product flags (health_score, priority_score, action_type) used ✅
No future-window data used ✅
Baseline is clean and honest.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.